

Главная идея эксперимента: модель не генерирует длинный ответ. Мы ограничиваем следующий токен двумя метками (намерениями|intent) `A` и `B`, получаем `log P(A)` и `log P(B)`, нормируем их внутри пары с помощью softmax и смотрим, как меняется решение при перестановке **семантики меток** и **их позиции**.

## 1. Схема эксперимента

![Описание картинки](attachments/pipeline.png)

Используются четыре контролируемых условия:

| Условие | Семантика A | Семантика B | Позиция A |
|---|---|---|---|
| E1 | CONTINUE | NEW | первая |
| E2 | NEW | CONTINUE | первая |
| E3 | CONTINUE | NEW | вторая |
| E4 | NEW | CONTINUE | вторая |

Это позволяет отделить собственно **семантический сигнал** от предпочтения конкретной метки `A/B`, позиции и их взаимодействия.

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_colwidth', 100)

DATA_DIR = Path('data')
FILES = {name: DATA_DIR / f'{name}.csv' for name in ['E1', 'E2', 'E3', 'E4']}

MAPPING = {
    'E1': {'A': 'CONTINUE', 'B': 'NEW'},
    'E2': {'A': 'NEW', 'B': 'CONTINUE'},
    'E3': {'A': 'CONTINUE', 'B': 'NEW'},
    'E4': {'A': 'NEW', 'B': 'CONTINUE'},
}

POSITION_A = {
    'E1': 'first',
    'E2': 'first',
    'E3': 'second',
    'E4': 'second',
}

runs = {name: pd.read_csv(path) for name, path in FILES.items()}
print({name: len(df) for name, df in runs.items()})

FileNotFoundError: [Errno 2] No such file or directory: 'data/E1.csv'

## 2. Что такое `logprobs`

LLM сначала строит распределение вероятностей следующего токена. Для двух допустимых вариантов можно записать:

$$
\ell_A = \log P(A), \qquad \ell_B = \log P(B).
$$

Чтобы получить удобные относительные вероятности внутри пары, используем устойчивую softmax-нормировку:

$$
P(A\mid A,B)=\frac{e^{\ell_A}}{e^{\ell_A}+e^{\ell_B}},
\qquad
P(B\mid A,B)=1-P(A\mid A,B).
$$

В исходных CSV эти величины уже сохранены как $p_a$ и $p_b$. Ниже проверяем, что они действительно суммируются в единицу.

In [ ]:
check = []
for condition, df in runs.items():
    pair_sum = df['p_a'] + df['p_b']
    check.append({
        'condition': condition,
        'min(PA+PB)': pair_sum.min(),
        'max(PA+PB)': pair_sum.max(),
        'mean(PA+PB)': pair_sum.mean(),
    })
display(pd.DataFrame(check))

## 3. Перевод токенов A/B в семантические намерения

Сравнивать просто `P(A)` между экспериментами нельзя: в E1/E3 токен `A` означает `CONTINUE`, а в E2/E4 — `NEW`.

Поэтому для каждого запуска переводим токеновую вероятность в:

- `p_continue`;
- `p_new`;
- семантический predicted route.

In [ ]:
rows = []
for condition, df in runs.items():
    mapping = MAPPING[condition]
    for _, row in df.iterrows():
        p_continue = row['p_a'] if mapping['A'] == 'CONTINUE' else row['p_b']
        p_new = row['p_a'] if mapping['A'] == 'NEW' else row['p_b']
        rows.append({
            'condition': condition,
            'id': row['id'],
            'difficulty': row['difficulty'],
            'message': row['message'],
            'expected': mapping[row['expected']],
            'predicted': mapping[row['predicted']],
            'correct': mapping[row['predicted']] == mapping[row['expected']],
            'p_continue': p_continue,
            'p_new': p_new,
            'logprob_a': row['logprob_a'],
            'logprob_b': row['logprob_b'],
            'd_ab': row['logprob_a'] - row['logprob_b'],
        })

semantic = pd.DataFrame(rows)
display(semantic.head(8))

## 4. Первый результат: точность очень зависит от представления

Если бы `logprobs` отражали только смысл `CONTINUE/NEW`, то перестановка меток и позиции не должна была бы радикально менять качество. Но это не так.

In [ ]:
summary = (
    semantic.groupby('condition')
    .agg(
        accuracy=('correct', 'mean'),
        mean_p_continue=('p_continue', 'mean'),
        mean_p_new=('p_new', 'mean'),
        n=('id', 'size'),
    )
    .reset_index()
)
summary['accuracy_pct'] = 100 * summary['accuracy']
display(summary[['condition', 'n', 'accuracy_pct', 'mean_p_continue', 'mean_p_new']])

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(summary['condition'], summary['accuracy_pct'])
plt.axhline(50, linestyle='--', linewidth=1)
plt.ylim(0, 100)
plt.ylabel('Accuracy, %')
plt.xlabel('Экспериментальное условие')
plt.title('Точность при разных кодировках и позициях маршрутов')
plt.show()

На этих данных:

- E1: около 90%;
- E2: 50% — модель выбирает `A`, которое в этом условии означает `NEW`, почти для всех случаев;
- E3/E4: около 93.3%.

Следовательно, высокая `P(A)` или `P(B)` сама по себе не является чистой вероятностью семантического намерения: в неё примешиваются эффекты представления.

## 5. Карта вероятности `CONTINUE` по всем случаям

Здесь каждая строка — один пример, каждый столбец — E1…E4. Значение — `P(CONTINUE)` после перевода токена в семантический маршрут.

Если модель была бы инвариантна к представлению, значения в одной строке были бы близкими между всеми четырьмя столбцами.

In [ ]:
prob_matrix = semantic.pivot(index='id', columns='condition', values='p_continue')[['E1', 'E2', 'E3', 'E4']]

plt.figure(figsize=(8, 10))
img = plt.imshow(prob_matrix.values, aspect='auto', vmin=0, vmax=1)
plt.colorbar(img, label='P(CONTINUE | два разрешённых маршрута)')
plt.xticks(range(4), prob_matrix.columns)
plt.yticks(range(len(prob_matrix)), prob_matrix.index, fontsize=7)
plt.xlabel('Условие')
plt.ylabel('Case')
plt.title('Чувствительность вероятности CONTINUE к представлению маршрутов')
plt.tight_layout()
plt.show()

## 6. Разбор одного случая

Функция ниже позволяет выбрать пример и увидеть, как менялась вероятность намерения при одной и той же семантической задаче.

In [ ]:
def show_case(case_id: str):
    cols = ['condition', 'expected', 'predicted', 'p_continue', 'p_new', 'd_ab', 'message']
    part = semantic[semantic['id'] == case_id][cols].copy()
    part['p_continue'] = part['p_continue'].round(6)
    part['p_new'] = part['p_new'].round(6)
    part['d_ab'] = part['d_ab'].round(4)
    display(part)

show_case('case_020')

### Почему это важно

Один и тот же пример может получать очень высокую вероятность одного токена, но после изменения соответствия `A/B` семантическим классам вывод меняется. Поэтому `confidence = max(P(A), P(B))` нельзя трактовать как вероятность правильности ответа.

## 7. Декомпозиция: семантика, label bias, position bias

Для каждого случая обозначим

\[
d_k = \log P(A)-\log P(B)
\]

для четырёх условий E1…E4. Тогда ортогональные контрасты дают:

\[
s=\frac{d_1-d_2+d_3-d_4}{4}
\]

— семантический сигнал `CONTINUE` против `NEW`;

\[
l=\frac{d_1+d_2+d_3+d_4}{4}
\]

— предпочтение метки `A` против `B`;

\[
p=\frac{d_1+d_2-d_3-d_4}{4}
\]

— позиционный эффект;

\[
i=\frac{d_1-d_2-d_3+d_4}{4}
\]

— взаимодействие семантики и позиции.

Это **диагностическая декомпозиция конкретного 2×2 эксперимента**, а не универсальные причинные константы модели.

In [ ]:
wide = {}
for condition, df in runs.items():
    temp = df.set_index('id').copy()
    temp['d_ab'] = temp['logprob_a'] - temp['logprob_b']
    wide[condition] = temp

records = []
for case_id in wide['E1'].index:
    d1 = wide['E1'].loc[case_id, 'd_ab']
    d2 = wide['E2'].loc[case_id, 'd_ab']
    d3 = wide['E3'].loc[case_id, 'd_ab']
    d4 = wide['E4'].loc[case_id, 'd_ab']

    s = (d1 - d2 + d3 - d4) / 4
    l = (d1 + d2 + d3 + d4) / 4
    p = (d1 + d2 - d3 - d4) / 4
    interaction = (d1 - d2 - d3 + d4) / 4

    expected = MAPPING['E1'][wide['E1'].loc[case_id, 'expected']]
    predicted = 'CONTINUE' if s > 0 else 'NEW'
    # Удобная диагностическая нормировка semantic contrast.
    p_continue_debiased = 1 / (1 + math.exp(-s)) if abs(s) < 700 else float(s > 0)

    records.append({
        'id': case_id,
        'expected': expected,
        'semantic_s': s,
        'label_bias_l': l,
        'position_bias_p': p,
        'interaction_i': interaction,
        'debiased_prediction': predicted,
        'p_continue_debiased': p_continue_debiased,
    })

decomp = pd.DataFrame(records)
decomp['correct'] = decomp['debiased_prediction'] == decomp['expected']
print(f"De-biased accuracy: {decomp['correct'].mean():.1%}")
display(decomp.head())

`sigmoid(s)` ниже используется только как удобная шкала от 0 до 1. Это **не калиброванная вероятность правильности** и не следует интерпретировать как «99.9% шанс, что реальный класс CONTINUE».

In [ ]:
plot_df = decomp.reset_index(drop=True)

plt.figure(figsize=(10, 5))
plt.scatter(np.arange(len(plot_df)), plot_df['semantic_s'])
plt.axhline(0, linestyle='--', linewidth=1)
plt.xticks(np.arange(len(plot_df)), plot_df['id'], rotation=90, fontsize=7)
plt.ylabel('Декомпозированный semantic signal s')
plt.xlabel('Case')
plt.title('Семантический сигнал после удаления label/position контрастов')
plt.tight_layout()
plt.show()

In [ ]:
errors = decomp[~decomp['correct']].copy()
errors['abs_s'] = errors['semantic_s'].abs()
display(errors.sort_values('abs_s', ascending=False)[
    ['id', 'expected', 'debiased_prediction', 'semantic_s', 'p_continue_debiased']
])

На текущих 30 случаях декомпозированный семантический сигнал даёт 28/30 = 93.3%. При этом две ошибки остаются с большим по модулю `s`: это показывает, что даже после устранения наблюдаемых смещений можно получить **уверенную семантическую ошибку**.

## 8. Что именно можно называть «вероятностью намерения»

В этой работе есть несколько разных величин, и их важно не смешивать:

1. **`P(A | A,B)` / `P(B | A,B)`** — относительная вероятность следующего label-токена при принудительном выборе только между A и B.
2. **`P(CONTINUE)` / `P(NEW)` в конкретном условии** — те же вероятности после перевода A/B в семантические маршруты.
3. **`sigmoid(s)`** — диагностически нормированный семантический контраст после 2×2 декомпозиции.
4. **Вероятность правильности маршрута** — отдельная величина. Ни один из трёх показателей выше автоматически ею не является; для этого нужна отдельная калибровка на независимой выборке.

Для диссертации корректнее говорить: **«оценка относительной вероятности выбора маршрута моделью»**, а не «вероятность того, что намерение пользователя действительно такое».

## 9. Дополнительный живой эксперимент через локальный vLLM

Ячейка ниже показывает минимальный API-пайплайн для нового сообщения. По умолчанию `RUN_LIVE = False`, поэтому notebook воспроизводится без запущенного сервера.

Для реального запуска нужен совместимый vLLM на `http://localhost:8000`, где токены `A` и `B` являются одиночными токенами модели.

In [ ]:
RUN_LIVE = False
BASE_URL = 'http://localhost:8000'
MODEL = None  # можно указать served model id вручную

In [ ]:
import requests

SYSTEM_PROMPT = """Ты выполняешь только маршрутизацию диалога.

Нужно определить, относится ли НОВОЕ сообщение пользователя к текущей задаче диалога.

Допустимые ветви:
A = CONTINUE
Новое сообщение продолжает текущую задачу: отвечает на вопрос ассистента,
уточняет её, меняет параметры или задаёт непосредственно связанный подвопрос.

B = NEW
Новое сообщение вводит новую независимую задачу, которую можно рассматривать
отдельно от текущей.

Если сообщение одновременно продолжает текущую тему и содержит новый независимый запрос, выбирай B.
Не решай запрос пользователя. Не объясняй решение. Выбери только A или B.
"""

def normalize_pair(log_a, log_b):
    m = max(log_a, log_b)
    ea = math.exp(log_a - m)
    eb = math.exp(log_b - m)
    z = ea + eb
    return ea / z, eb / z


def live_route(history, message, base_url=BASE_URL, model=MODEL):
    session = requests.Session()
    if model is None:
        models = session.get(f'{base_url}/v1/models', timeout=30).json()['data']
        model = models[0]['id']

    def token_id(label):
        r = session.post(
            f'{base_url}/tokenize',
            json={'model': model, 'prompt': label, 'add_special_tokens': False},
            timeout=30,
        )
        r.raise_for_status()
        tokens = r.json()['tokens']
        if len(tokens) != 1:
            raise ValueError(f'{label!r} tokenizes into {tokens}, expected one token')
        return tokens[0]

    token_a, token_b = token_id('A'), token_id('B')
    rendered = '\n'.join(
        f"{'Пользователь' if x['role']=='user' else 'Ассистент'}: {x['content']}"
        for x in history
    )
    user_prompt = (
        f'ТЕКУЩИЙ ДИАЛОГ:\n{rendered}\n\n'
        f'НОВОЕ СООБЩЕНИЕ ПОЛЬЗОВАТЕЛЯ:\n{message}\n\n'
        'Выбери маршрут A или B.'
    )

    payload = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
        'max_completion_tokens': 1,
        'temperature': 0,
        'seed': 0,
        'logprobs': True,
        'top_logprobs': 0,
        'logprob_token_ids': [token_a, token_b],
        'allowed_token_ids': [token_a, token_b],
        'chat_template_kwargs': {'enable_thinking': False},
    }

    r = session.post(f'{base_url}/v1/chat/completions', json=payload, timeout=60)
    r.raise_for_status()
    result = r.json()
    pos = result['choices'][0]['logprobs']['content'][0]

    candidates = {pos.get('token', '').strip(): pos.get('logprob')}
    for item in pos.get('top_logprobs') or []:
        candidates[item['token'].strip()] = item['logprob']

    log_a, log_b = float(candidates['A']), float(candidates['B'])
    p_a, p_b = normalize_pair(log_a, log_b)
    return {
        'model': model,
        'logP(A)': log_a,
        'logP(B)': log_b,
        'P(CONTINUE)': p_a,
        'P(NEW)': p_b,
        'route': 'CONTINUE' if p_a >= p_b else 'NEW',
    }

if RUN_LIVE:
    demo = live_route(
        history=[
            {'role': 'user', 'content': 'Сравни продажи Москвы и Петербурга за январь.'},
            {'role': 'assistant', 'content': 'Хорошо, подготовлю сравнение.'},
        ],
        message='Замени Петербург на Казань за тот же период.',
    )
    display(pd.DataFrame([demo]))
else:
    print('Live-запуск выключен. Установите RUN_LIVE = True при запущенном локальном vLLM.')

## 10. Итог

Показанный эксперимент отвечает на два разных вопроса:

1. **Как извлечь численную склонность LLM выбрать один маршрут вместо другого?**  
   Через `logprobs` разрешённых label-токенов и их нормировку.

2. **Можно ли считать эту величину надёжной вероятностью намерения?**  
   Нет без дополнительных проверок. Наш 2×2 эксперимент показал, что на оценку сильно влияют label и позиция кандидата.

Главный исследовательский результат этого notebook:

\[
\boxed{\text{token probability} \neq \text{calibrated semantic correctness probability}}
\]

Поэтому `logprobs` полезны как диагностический инструмент и как источник количественного сигнала, но для надёжного роутинга необходимо отдельно контролировать устойчивость к представлению и проверять калибровку на независимом наборе данных.